# 💻 LPARA End-to-End Pipeline: Raw 1K Dataset Cleaning & Multi-Algorithm ML Training
### From Raw Scraped Laptops (1K Records) ➡️ Automated Cleaning & Imputation ➡️ Feature Engineering ➡️ Model Benchmarking & Stacking Ensemble

This notebook guides you from **raw 1,000+ scraped laptop records** all the way to a production-grade regression model:
1. **Environment Setup & Imports**
2. **Load Raw 1K Dataset** (`laptop_prices_complete_1k.csv`)
3. **Audit Missing Values & Data Quality Issues**
4. **Step-by-Step Data Cleaning Pipeline**:
   - Target price unification & sanity filtering (₹10,000 to ₹10,00,000)
   - Filter out accessories & non-laptop items
   - Brand name normalization
   - RAM & Storage type cleaning & numeric casting
   - Hardware-aware OS & GPU imputation
5. **Domain Feature Engineering**:
   - CPU Performance Tiering (Tiers 1-5)
   - Touchscreen, OLED, Gaming & Apple flags
   - RAM-to-Storage ratio
6. **Deduplication & Final Zero-Missing Audit**
7. **Multi-Algorithm Regression Benchmark** (8 Models across 5-Fold CV & 20% Holdout Test)
8. **Visualizations** (Missing Value Heatmap, Price Distributions, R² Leaderboard, 1:1 Fit Plot)
9. **Save Winning Stacking Ensemble Model**
10. **Interactive Fair Market Price Predictor**


In [ ]:
# Cell 1: Install & Import Dependencies
!pip install -q scikit-learn xgboost pandas numpy joblib matplotlib seaborn

import os
import re
import sys
import time
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    StackingRegressor
)
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print('✅ All libraries successfully imported!')


In [ ]:
# Cell 2: Load the Raw 1K Dataset
from google.colab import files

raw_file = 'laptop_prices_complete_1k.csv'
if not os.path.exists(raw_file):
    print('Please upload laptop_prices_complete_1k.csv from your local machine:')
    uploaded = files.upload()

raw_df = pd.read_csv(raw_file)
print(f'✅ Loaded Raw Dataset: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
display(raw_df.head(3))


In [ ]:
# Cell 3: Initial Data Audit (Missing Values in Raw Data)
print('--- RAW DATASET QUALITY AUDIT ---')
missing_counts = raw_df.isna().sum()
missing_pct = (missing_counts / len(raw_df) * 100).round(1)
audit_df = pd.DataFrame({'Missing_Count': missing_counts, 'Missing_Percent': missing_pct})
audit_df = audit_df[audit_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

print(f'Columns with missing values: {len(audit_df)} / {raw_df.shape[1]}')
display(audit_df.head(15))


In [ ]:
# Cell 4: Cleaning Functions Definition
def clean_ram_type(rt):
    rt_u = str(rt or '').upper()
    if 'LPDDR5X' in rt_u: return 'LPDDR5X'
    if 'LPDDR5' in rt_u: return 'LPDDR5'
    if 'DDR5' in rt_u: return 'DDR5'
    if 'LPDDR4X' in rt_u: return 'LPDDR4X'
    if 'DDR4' in rt_u: return 'DDR4'
    if 'UNIFIED' in rt_u: return 'Unified Memory'
    return 'DDR5'

def clean_storage_type(st):
    st_u = str(st or '').upper()
    if 'NVME' in st_u: return 'NVMe SSD'
    if 'PCIE' in st_u: return 'PCIe SSD'
    if 'EMMC' in st_u: return 'eMMC'
    if 'HDD' in st_u: return 'HDD'
    return 'SSD'

def normalize_brand(b):
    brand_clean = str(b or '').strip()
    brand_map = {
        'asus': 'ASUS', 'hp': 'HP', 'dell': 'Dell', 'lenovo': 'Lenovo',
        'acer': 'Acer', 'msi': 'MSI', 'apple': 'Apple', 'samsung': 'Samsung',
        'microsoft': 'Microsoft', 'infinix': 'Infinix', 'gigabyte': 'Gigabyte',
        'motorola': 'Motorola', 'zebronics': 'Zebronics', 'colorful': 'Colorful'
    }
    return brand_map.get(brand_clean.lower(), brand_clean)

def extract_cpu_tier(cpu_model, cpu_family):
    text = (str(cpu_family or '') + ' ' + str(cpu_model or '')).lower()
    if any(k in text for k in ['celeron', 'pentium', 'athlon', 'kompanio', 'n4020', 'n4500']):
        return 1
    if any(k in text for k in ['core i3', 'i3-', 'ryzen 3', 'core 3']):
        return 2
    if any(k in text for k in ['core i5', 'i5-', 'ryzen 5', 'core 5', 'apple m1', 'm1 ']):
        return 3
    if any(k in text for k in ['core i7', 'i7-', 'ryzen 7', 'core 7', 'core ultra 5', 'apple m2', 'apple m3', 'snapdragon x']):
        return 4
    if any(k in text for k in ['core i9', 'i9-', 'ryzen 9', 'core 9', 'core ultra 7', 'core ultra 9', 'm2 max', 'm3 max', 'm4', 'threadripper', 'ai max']):
        return 5
    return 3

def assign_price_category(price):
    if price < 40000: return 'Budget (<40K)'
    elif price < 75000: return 'Mid-Range (40K-75K)'
    elif price < 125000: return 'Upper Mid-Range (75K-1.25L)'
    else: return 'Premium / Flagship (>1.25L)'

print('✅ Cleaning helper functions configured!')


In [ ]:
# Cell 5: Execute Data Cleaning Pipeline on 1K Records
accessory_keywords = [
    'screen guard', 'screen protector', 'keyboard replacement',
    'adapter', 'charger', 'battery', 'laptop skin', 'cable', 'mouse pad'
]

cleaned_records = []
raw_records = raw_df.to_dict(orient='records')

for r in raw_records:
    # 1. Unify Target Price
    price = r.get('price_average') or r.get('price_current')
    if not price:
        continue
    try:
        price = float(price)
    except (ValueError, TypeError):
        continue

    # Sanity bounds: ignore price outliers / erroneous test listings
    if price < 10000 or price > 1000000:
        continue

    # 2. Filter non-laptop accessory records
    model_name = str(r.get('model') or '').strip()
    if any(ak in model_name.lower() for ak in accessory_keywords):
        continue

    # 3. Filter missing core identifiers
    brand = normalize_brand(r.get('brand'))
    if not brand or brand in ['Unknown', 'None', '']:
        continue

    cpu_brand = str(r.get('cpu_brand') or '').strip()
    cpu_model = str(r.get('cpu_model') or '').strip()
    ram_gb = r.get('ram_gb')
    storage_gb = r.get('storage_gb')
    display_size = r.get('display_size_inches')

    if not (cpu_brand and cpu_model and ram_gb and storage_gb and display_size):
        continue

    try:
        ram_gb = int(round(float(ram_gb)))
        storage_gb = int(round(float(storage_gb)))
        display_size = round(float(display_size), 1)
    except (ValueError, TypeError):
        continue

    # Valid hardware ranges
    if ram_gb not in [4, 8, 12, 16, 24, 32, 36, 48, 64, 128]:
        continue
    if storage_gb not in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
        continue
    if display_size < 10.0 or display_size > 18.0:
        continue

    # 4. Standardize GPU & OS
    gpu_t = str(r.get('gpu_type') or '').strip()
    if not gpu_t or gpu_t == 'nan':
        gpu_t = 'Integrated'

    os_str = str(r.get('operating_system') or '').strip()
    if not os_str or os_str == 'nan':
        os_str = 'macOS' if brand == 'Apple' else 'Windows 11 Home'

    cpu_fam = str(r.get('cpu_family') or '').strip()
    if not cpu_fam or cpu_fam == 'nan':
        cpu_fam = cpu_model

    cleaned_records.append({
        'brand': brand,
        'model': model_name,
        'price_average': round(price, 2),
        'cpu_brand': cpu_brand,
        'cpu_model': cpu_model,
        'cpu_family': cpu_fam,
        'ram_gb': ram_gb,
        'ram_type': clean_ram_type(r.get('ram_type')),
        'storage_gb': storage_gb,
        'storage_type': clean_storage_type(r.get('storage_type')),
        'display_size_inches': display_size,
        'gpu_type': gpu_t,
        'operating_system': os_str
    })

df_clean = pd.DataFrame(cleaned_records)
print(f'1. Validated & Standardized Candidates: {len(df_clean)}')

# 5. Deduplicate on composite hardware configuration
dedup_cols = ['brand', 'model', 'cpu_model', 'ram_gb', 'storage_gb', 'display_size_inches']
df_clean.drop_duplicates(subset=dedup_cols, inplace=True)
print(f'2. After Deduplication: {len(df_clean)} unique configurations')

# 6. Feature Engineering
model_lower = df_clean['model'].str.lower()
df_clean['is_apple'] = (df_clean['brand'] == 'Apple').astype(int)
df_clean['is_gaming'] = (
    (df_clean['gpu_type'] == 'Discrete') |
    model_lower.str.contains(r'tuf|rog|legion|loq|victus|omen|nitro|predator|katana|cyborg|sword|bravo|alienware|gaming|g15', regex=True)
).astype(int)
df_clean['is_touchscreen'] = model_lower.str.contains(r'touch|flip|2-in-1|360|yoga', regex=True).astype(int)
df_clean['is_oled'] = model_lower.str.contains('oled', regex=True).astype(int)
df_clean['cpu_tier'] = [extract_cpu_tier(m, f) for m, f in zip(df_clean['cpu_model'], df_clean['cpu_family'])]
df_clean['ram_storage_ratio'] = (df_clean['ram_gb'] / df_clean['storage_gb']).round(4)
df_clean['price_category'] = df_clean['price_average'].apply(assign_price_category)

print(f'3. Zero-Missing Audit: Total Nulls across all cells = {df_clean.isna().sum().sum()}')
display(df_clean.head(4))


In [ ]:
# Cell 6: Export Cleaned Dataset for Future Reuse
df_clean.to_csv('cleaned_laptop_dataset.csv', index=False)
print(f'✅ Successfully exported cleaned_laptop_dataset.csv ({len(df_clean)} rows, {len(df_clean.columns)} columns)')


In [ ]:
# Cell 7: Feature Pipeline Architecture (Scaling, Encoding & Statistical Feature Selection)
from sklearn.base import BaseEstimator, TransformerMixin

SELECTED_NUMERIC_FEATURES = [
    'ram_gb',
    'storage_gb',
    'display_size_inches',
    'ram_storage_ratio',
    'cpu_tier',
    'is_gaming',
    'is_apple'
]

SELECTED_CATEGORICAL_FEATURES = [
    'brand',
    'cpu_brand',
    'cpu_family',
    'ram_type',
    'storage_type',
    'gpu_type',
    'operating_system'
]

class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df_out = X.copy()
        if not isinstance(df_out, pd.DataFrame):
            df_out = pd.DataFrame(df_out)

        model_str = df_out.get('model', pd.Series([''] * len(df_out))).astype(str).str.lower()
        brand_str = df_out.get('brand', pd.Series([''] * len(df_out))).astype(str)

        df_out['is_apple'] = (brand_str == 'Apple').astype(int)
        gpu_type_str = df_out.get('gpu_type', pd.Series(['Integrated'] * len(df_out))).astype(str)
        is_discrete = (gpu_type_str == 'Discrete')
        gaming_kw = model_str.str.contains(
            r'tuf|rog|legion|loq|victus|omen|nitro|predator|katana|cyborg|sword|bravo|alienware|gaming|g15',
            regex=True
        )
        df_out['is_gaming'] = (is_discrete | gaming_kw).astype(int)

        cpu_models = df_out.get('cpu_model', pd.Series([''] * len(df_out))).astype(str)
        cpu_families = df_out.get('cpu_family', pd.Series([''] * len(df_out))).astype(str)
        df_out['cpu_tier'] = [extract_cpu_tier(m, f) for m, f in zip(cpu_models, cpu_families)]

        ram = pd.to_numeric(df_out.get('ram_gb', 16), errors='coerce').fillna(16)
        storage = pd.to_numeric(df_out.get('storage_gb', 512), errors='coerce').fillna(512).replace(0, 512)
        df_out['ram_storage_ratio'] = ram / storage

        for c in SELECTED_CATEGORICAL_FEATURES:
            if c not in df_out.columns:
                df_out[c] = 'Unknown'
            else:
                df_out[c] = df_out[c].fillna('Unknown').astype(str)

        for n in SELECTED_NUMERIC_FEATURES:
            if n not in df_out.columns:
                df_out[n] = 0.0
            else:
                df_out[n] = pd.to_numeric(df_out[n], errors='coerce').fillna(0.0)

        return df_out[SELECTED_NUMERIC_FEATURES + SELECTED_CATEGORICAL_FEATURES]

def build_preprocessor(percentile=85):
    numeric_pipe = Pipeline([('scaler', StandardScaler())])
    categorical_pipe = Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    col_tf = ColumnTransformer([
        ('num', numeric_pipe, SELECTED_NUMERIC_FEATURES),
        ('cat', categorical_pipe, SELECTED_CATEGORICAL_FEATURES)
    ])
    return Pipeline([
        ('feat_eng', FeatureEngineeringTransformer()),
        ('encoder', col_tf),
        ('selector', SelectPercentile(score_func=f_regression, percentile=percentile))
    ])

print('✅ Preprocessor and feature selector configured!')


In [ ]:
# Cell 8: Multi-Algorithm Benchmark (8 Models across 5-Fold CV + 20% Test)
target_col = 'price_average'
y = df_clean[target_col].values
drop_cols = [c for c in [target_col, 'price_category'] if c in df_clean.columns]
X = df_clean.drop(columns=drop_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples\n')

base_ridge = Ridge(alpha=1.0)
base_rf = RandomForestRegressor(n_estimators=180, max_depth=14, min_samples_split=3, random_state=42, n_jobs=-1)
base_gb = GradientBoostingRegressor(n_estimators=180, learning_rate=0.06, max_depth=5, random_state=42)
base_xgb = xgb.XGBRegressor(n_estimators=180, learning_rate=0.06, max_depth=5, random_state=42, n_jobs=-1)

stacking_ensemble = StackingRegressor(
    estimators=[
        ('rf', base_rf),
        ('gb', base_gb),
        ('xgb', base_xgb),
        ('ridge', base_ridge)
    ],
    final_estimator=Ridge(alpha=0.5),
    n_jobs=1
)

candidates = {
    'Ridge Regression': base_ridge,
    'Lasso Regression': Lasso(alpha=0.001, max_iter=2000),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_split=4, random_state=42),
    'Random Forest': base_rf,
    'Extra Trees': ExtraTreesRegressor(n_estimators=180, max_depth=14, min_samples_split=3, random_state=42, n_jobs=-1),
    'Gradient Boosting': base_gb,
    'XGBoost Regressor': base_xgb,
    'Stacking Ensemble': stacking_ensemble
}

benchmark_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Algorithm':<22} | {'CV R² (5-Fold)':<14} | {'Test R²':<10} | {'Test MAE':<14} | {'Test MAPE':<10}")
print('-' * 80)

best_model_name = None
best_pipeline = None
best_test_r2 = -float('inf')

for name, regressor in candidates.items():
    t0 = time.time()
    preprocessor = build_preprocessor(percentile=85)
    wrapped_model = TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', wrapped_model)
    ])

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2', n_jobs=1)
    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    test_r2 = float(r2_score(y_test, y_pred))
    test_mae = float(mean_absolute_error(y_test, y_pred))
    test_rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    test_mape = float(np.mean(np.abs((y_test - y_pred) / y_test)) * 100)
    elapsed = time.time() - t0

    badge = ' 🏆' if test_r2 > 0.86 else '   '
    print(f"{name:<22} | {cv_mean:.4f} ±{cv_std:.3f} | {test_r2:.4f}{badge} | ₹{test_mae:>10,.2f}  | {test_mape:>7.2f}%")

    benchmark_results.append({
        'Algorithm': name,
        'CV_R2': cv_mean,
        'CV_R2_Std': cv_std,
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_RMSE': test_rmse,
        'Test_MAPE': test_mape,
        'Time_s': round(elapsed, 2)
    })

    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        best_model_name = name
        best_pipeline = pipe

results_df = pd.DataFrame(benchmark_results).sort_values(by='Test_R2', ascending=False)
print('-' * 80)
print(f"\n🏆 WINNING MODEL: '{best_model_name}' (Test R² = {best_test_r2:.4f})")


In [ ]:
# Cell 9: Visualizations (Leaderboard & 1:1 Fit)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=results_df, x='Test_R2', y='Algorithm', palette='Blues_r', ax=axes[0])
axes[0].set_title('Test R² Score Comparison Across Algorithms', fontsize=13, fontweight='bold')
axes[0].set_xlim(0.65, 0.90)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.4f}", (p.get_width() + 0.003, p.get_y() + p.get_height() / 2),
                     va='center', fontsize=10, fontweight='bold')

y_best_pred = best_pipeline.predict(X_test)
axes[1].scatter(y_test, y_best_pred, alpha=0.6, color='#4f46e5', edgecolors='k', s=45)
max_val = max(np.max(y_test), np.max(y_best_pred))
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect 1:1 Fit')
axes[1].set_title(f'Actual vs Predicted Prices ({best_model_name})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual Price (₹ INR)', fontsize=11)
axes[1].set_ylabel('Predicted Price (₹ INR)', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Cell 10: Save Best Model Artifact
output_model_path = 'best_laptop_price_model.joblib'
joblib.dump(best_pipeline, output_model_path)
print(f'✅ Successfully saved best model pipeline to: {output_model_path}')


In [ ]:
# Cell 11: Interactive Prediction Function
def predict_price(specs_dict):
    df_single = pd.DataFrame([specs_dict])
    pred = float(best_pipeline.predict(df_single)[0])
    pred = max(10000.0, round(pred, 2))
    low = round(pred * 0.91, 2)
    high = round(pred * 1.09, 2)
    return {
        'Predicted Price': f'₹{pred:,.2f}',
        'Fair Market Band (±9%)': f'₹{low:,.2f} – ₹{high:,.2f}'
    }

# Test with a sample laptop
sample = {
    'brand': 'ASUS',
    'model': 'Zenbook 14 OLED',
    'cpu_brand': 'Intel',
    'cpu_model': 'Core Ultra 7 155H',
    'cpu_family': 'Core Ultra 7',
    'ram_gb': 16,
    'ram_type': 'LPDDR5X',
    'storage_gb': 1024,
    'storage_type': 'NVMe SSD',
    'display_size_inches': 14.0,
    'gpu_type': 'Integrated',
    'operating_system': 'Windows 11 Home'
}

res = predict_price(sample)
print('--- FAIR MARKET PREDICTION DEMO ---')
print(f"Laptop: {sample['brand']} {sample['model']}")
print(f"Specs:  {sample['cpu_model']} | {sample['ram_gb']}GB RAM | {sample['storage_gb']}GB SSD")
print(f"Predicted Value: {res['Predicted Price']}")
print(f"Fair Range:      {res['Fair Market Band (±9%)']}")
